In [1]:
import os
from typing import List, TypedDict, Union

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import END, START, StateGraph

load_dotenv()

True

In [2]:
if "GROQ_API_KEY" not in os.environ:
    raise RuntimeError("GROQ_API_KEY is not set. Add it to your .env file or environment.")


class AgentState(TypedDict):
    messages: List[Union[HumanMessage, AIMessage]]


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)


def process(state: AgentState) -> AgentState:
    """Run the Groq-backed agent node."""
    response = llm.invoke(state["messages"])

    state["messages"].append(AIMessage(content=response.content))
    print(f"\nAI: {response.content}")
    print("CURRENT STATE:", state["messages"])

    return state


In [3]:


graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END)
agent = graph.compile()

conversation_history = []

user_input = input("Enter: ")
while user_input != "exit":
    conversation_history.append(HumanMessage(content=user_input))
    result = agent.invoke({"messages": conversation_history})
    conversation_history = result["messages"]
    print("CURRENT CONVERSATION STATE:", conversation_history)
    user_input = input("Enter: ")


AI: It's nice to meet you. Is there something I can help you with or would you like to chat?
CURRENT STATE: [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
CURRENT CONVERSATION STATE: [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

AI: Hi again. How's your day going so far? Want to talk about something in particular or just say hello?
CURRENT STATE: [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, respo